# Minimal example
Run the size mitigation on a handful of spws in one call and show the final mitigation table.

Requires `smart_size_mitigation_heuristics.py` in the same directory (or on PYTHONPATH).

In [19]:
from smart_size_mitigation_heuristics import (
    build_mitigation_table,
    sizes_from_imlist,
    Limits,
    plan,
)

# 1. Define the input parameters (example case)

In [20]:
spws = [
    {  # representative spw: nbin-eligible, single pointing, robust=2.0
        "spw": "25", "nchan": 3840, "ratio_satisfied": False,
        "pb": 0.9, "is_mosaic": False, "ppb": 4.0, "robust": 2.0,
        "n_roi": 3000, "n_total": 3840,
    },
    {  # smaller nchan, only eligible for nbin if ratio condition met
        "spw": "27", "nchan": 960, "ratio_satisfied": True,
        "pb": 1.0, "is_mosaic": False, "ppb": 3.5, "robust": None,
        "n_roi": 800, "n_total": 960,
    },
    {  # nbin-eligible (nchan=7680), single pointing
        "spw": "29", "nchan": 7680, "ratio_satisfied": False,
        "pb": 0.8, "is_mosaic": False, "ppb": 4.0, "robust": None,
        "n_roi": 1000, "n_total": 7680,
    },
    {  # not nbin-eligible at all (ratio condition not met), tight PB already
        "spw": "31", "nchan": 1920, "ratio_satisfied": False,
        "pb": 0.65, "is_mosaic": False, "ppb": 5.0, "robust": 2.0,
        "n_roi": 1200, "n_total": 1920,
    },
]

# The sizes are not typed in: they are computed from the imaging target list
# (in the pipeline: the targets of makeimlist's result) with calculate_sizes().
# Here: 3 fields, each imaged in every spw.
imsizes = {"25": 990, "27": 1980, "29": 1510, "31": 1400}
imlist = [
    {"field": field, "spw": spw, "imsize": (n, n), "nbin": -1, "specmode": "cube"}
    for field in ("f1", "f2", "f3")
    for spw, n in imsizes.items()
]
get_nchan = {row["spw"]: row["nchan"] for row in spws}.__getitem__  # spw -> nchan

# 2. Compute the sizes and build the per-lever mitigation table

In [21]:
cube_sizes, product_sizes, nfields = sizes_from_imlist(imlist, get_nchan)
print("cube sizes (GB):   ", {k: round(v, 1) for k, v in cube_sizes.items()})
print("product sizes (GB):", {k: round(v, 1) for k, v in product_sizes.items()})
print("nfields:", nfields)

table = build_mitigation_table(spws, use_roi=False, nfields=nfields)
table

cube sizes (GB):    {'25': 15.1, '27': 15.1, '29': 70.0, '31': 15.1}
product sizes (GB): {'25': 90.3, '27': 90.4, '29': 420.3, '31': 90.4}
nfields: 3


,n_bin,PB,cell,ROI,nfields_total,nfields_kept,nfields,total_reduction
spw,,,,,,,,
25,0.5,0.395062,0.339844,0.0,3,3,0.666667,0.933441
27,0.5,0.510000,0.265306,0.0,3,3,0.666667,0.940000
29,0.5,0.234375,0.437500,0.0,3,3,0.666667,0.928223
31,0.0,0.000000,0.577500,0.0,3,3,0.666667,0.859167


# 3. Set the size limits (GB)

In [22]:
limits = Limits(maxcubesize=10.0, maxproductsize=60.0, maxcubelimit=50.0)
limits


Limits(maxcubesize=10.0, maxproductsize=60.0, maxcubelimit=50.0)

# 4. Run the mitigation in one call. spw "29" is the representative spw here
Every possible plan is scored and the one with the highest score is returned. `return_candidates=True` additionally returns the full list of plans (step 6).

In [23]:
final, msg = plan(
    table,
    limits,
    imlist,
    get_nchan,
    repr_spw="29",
    return_candidates=True,  # for the diagnostic list below
)

# 5. Display the outcome.

In [24]:
print(f"Result: {msg['shortmsg']}")
print(f"\nFinal score: {msg['score']:.3f}  (1 = no mitigation needed, 0 = limits cannot be met)")
print(msg["scores"].round(3).to_string())
print(f"\nSpws kept: {list(final.index[final['kept']])}")

Result: Size was mitigated

Final score: 0.177  (1 = no mitigation needed, 0 = limits cannot be met)
n_bin     0.950
PB        0.972
cell      0.959
ROI       1.000
spws      1.000
fields    0.200

Spws kept: ['25', '27', '29', '31']


In [25]:
final

,n_bin,PB,cell,ROI,nfields_total,nfields_kept,nfields,total_reduction,cube_size_GB,product_size_GB,required_reduction,min_steps,levers_used,achieved_reduction,field_reduction,mitigated_cube_GB,mitigated_product_GB,kept
spw,,,,,,,,,,,,,,,,,,
25,0.5,0.395062,0.339844,0.0,3,1,0.666667,0.933441,15.054336,90.349538,0.335740,4,"n_bin,PB,cell,nfields",0.800323,0.666667,3.006003,6.013572,True
27,0.5,0.510000,0.265306,0.0,3,1,0.666667,0.940000,15.054336,90.420106,0.335740,3,"PB,cell,nfields",0.640000,0.666667,5.419561,10.850413,True
29,0.5,0.234375,0.437500,0.0,3,1,0.666667,0.928223,70.044672,420.322754,0.857234,4,"n_bin,PB,cell,nfields",0.784668,0.666667,15.082862,30.169651,True
31,0.0,0.000000,0.577500,0.0,3,1,0.666667,0.859167,15.052800,90.363840,0.335672,2,"cell,nfields",0.577500,0.666667,6.359808,12.726241,True


# 6. Diagnostics: all plans, sorted by score
One row per plan: the cube levers applied to each spw (`dropped` = spw removed), fields kept, resulting sizes (GB), whether all limits are met, the lever scores and the final `score` (0 if infeasible).

In [26]:
candidates = msg["candidates"]
print(f"{len(candidates)} plans, {candidates['feasible'].sum()} feasible")
candidates.head(15).round(3)

336 plans, 68 feasible


,29,25,27,31,n_spws_kept,nfields_kept,n_levers,product_GB,max_cube_GB,feasible,n_bin,PB,cell,ROI,spws,fields,score
0,"n_bin,PB,cell","n_bin,PB,cell","PB,cell",cell,4,1,9,59.760,15.083,True,0.950,0.972,0.959,1.0,1.000,0.2,0.177
1,"n_bin,PB,cell","n_bin,PB","n_bin,PB",cell,4,1,8,59.390,15.083,True,0.925,0.972,0.975,1.0,1.000,0.2,0.175
2,"n_bin,PB,cell","n_bin,PB,cell","n_bin,cell",cell,4,1,9,59.981,15.083,True,0.925,0.984,0.959,1.0,1.000,0.2,0.175
3,"n_bin,PB,cell","n_bin,cell","n_bin,PB,cell",cell,4,1,9,58.262,15.083,True,0.925,0.981,0.959,1.0,1.000,0.2,0.174
4,"n_bin,PB,cell","n_bin,PB","n_bin,PB,cell",cell,4,1,9,57.430,15.083,True,0.925,0.972,0.968,1.0,1.000,0.2,0.174
5,"n_bin,PB,cell","n_bin,PB,cell","n_bin,PB",cell,4,1,9,56.294,15.083,True,0.925,0.972,0.966,1.0,1.000,0.2,0.174
6,"n_bin,PB,cell","n_bin,PB,cell","n_bin,PB,cell",cell,4,1,10,54.335,15.083,True,0.925,0.972,0.959,1.0,1.000,0.2,0.172
7,"n_bin,PB,cell",dropped,PB,cell,3,1,5,57.665,15.083,True,0.967,0.975,0.966,1.0,0.533,0.2,0.097
8,"n_bin,PB,cell",PB,"PB,cell",dropped,3,1,6,59.239,15.083,True,0.967,0.962,0.977,1.0,0.533,0.2,0.097
9,"n_bin,PB,cell","PB,cell",PB,dropped,3,1,6,56.965,15.083,True,0.967,0.962,0.974,1.0,0.533,0.2,0.097
